# 🐍 Day 1 — Advanced OOP & Design Patterns
### 15-Day Intermediate Python Course

> **Goal:** Master Python's OOP system beyond the basics — ABCs, MRO, descriptors, dataclasses, and production-grade design patterns.

---
**Topics:**
1. Abstract Base Classes (`abc` module)
2. MRO & Multiple Inheritance
3. Descriptors, `__slots__`, `__init_subclass__`
4. Design Patterns: Singleton, Factory, Observer
5. Dataclasses — advanced usage
6. ✅ Quiz
7. 💼 Interview Questions & Answers
8. 🔨 Real-World Use Case
9. 🚀 Mini Project

---

## 1. Abstract Base Classes (`abc` module)

An **Abstract Base Class** defines a *contract* — a set of methods subclasses MUST implement.  
It prevents instantiation of incomplete classes at runtime.


In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    """Abstract base class for all shapes."""

    @abstractmethod
    def area(self) -> float:
        """Every shape must implement area()."""
        ...

    @abstractmethod
    def perimeter(self) -> float:
        """Every shape must implement perimeter()."""
        ...

    def describe(self) -> str:
        """Concrete method shared by all shapes."""
        return f"{self.__class__.__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}"


class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius

    def area(self) -> float:
        import math
        return math.pi * self.radius ** 2

    def perimeter(self) -> float:
        import math
        return 2 * math.pi * self.radius


class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height

    def perimeter(self) -> float:
        return 2 * (self.width + self.height)


# Usage
c = Circle(5)
r = Rectangle(4, 6)
print(c.describe())
print(r.describe())

# Trying to instantiate Shape directly raises TypeError
try:
    s = Shape()
except TypeError as e:
    print(f"\nCan't instantiate ABC: {e}")


In [ ]:
# ABCs can also be used for isinstance() checks — even without inheritance
# by registering virtual subclasses

from abc import ABC, abstractmethod

class Drawable(ABC):
    @abstractmethod
    def draw(self): ...

class LegacySVG:
    """Old class we can't modify, but it has a draw() method."""
    def draw(self):
        return "<svg>...</svg>"

# Register as virtual subclass
Drawable.register(LegacySVG)

svg = LegacySVG()
print(isinstance(svg, Drawable))   # True — even without inheritance!
print(issubclass(LegacySVG, Drawable))  # True


## 2. Method Resolution Order (MRO) & Multiple Inheritance

Python uses the **C3 Linearization algorithm** to determine the order in which classes are searched  
when resolving method calls. Understanding MRO prevents subtle bugs in multiple inheritance.


In [ ]:
# Visualising MRO with a diamond inheritance
class A:
    def greet(self):
        return "Hello from A"

class B(A):
    def greet(self):
        return "Hello from B"

class C(A):
    def greet(self):
        return "Hello from C"

class D(B, C):
    pass  # Inherits from both B and C

d = D()
print(d.greet())          # Which greet() wins?
print(D.__mro__)          # Full MRO chain
print([cls.__name__ for cls in D.__mro__])  # Readable form


In [ ]:
# Using super() correctly with MRO — cooperative multiple inheritance
class Base:
    def __init__(self):
        print("Base.__init__")
        super().__init__()

class Logging:
    def __init__(self):
        print("Logging.__init__")
        super().__init__()

class Serializable:
    def __init__(self):
        print("Serializable.__init__")
        super().__init__()

class Model(Logging, Serializable, Base):
    def __init__(self):
        print("Model.__init__")
        super().__init__()

print("MRO:", [c.__name__ for c in Model.__mro__])
print()
m = Model()
# Notice how super() chains through every class in MRO order


## 3. Descriptors, `__slots__`, and `__init_subclass__`

### Descriptors
A **descriptor** is any object that defines `__get__`, `__set__`, or `__delete__`.  
This is how Python implements properties, classmethods, staticmethods under the hood.


In [ ]:
# Building a custom validated descriptor
class PositiveNumber:
    """Descriptor that enforces positive numeric values."""

    def __set_name__(self, owner, name):
        self.name = name
        self.private_name = f"_{name}"

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.private_name, None)

    def __set__(self, obj, value):
        if not isinstance(value, (int, float)):
            raise TypeError(f"{self.name} must be a number, got {type(value).__name__}")
        if value <= 0:
            raise ValueError(f"{self.name} must be positive, got {value}")
        setattr(obj, self.private_name, value)


class Product:
    price = PositiveNumber()
    stock = PositiveNumber()

    def __init__(self, name: str, price: float, stock: int):
        self.name = name
        self.price = price
        self.stock = stock

    def __repr__(self):
        return f"Product({self.name!r}, price={self.price}, stock={self.stock})"


p = Product("Widget", 9.99, 100)
print(p)

# Descriptor enforces constraints
try:
    p.price = -5
except ValueError as e:
    print(f"ValueError: {e}")

try:
    p.stock = "lots"
except TypeError as e:
    print(f"TypeError: {e}")


In [ ]:
# __slots__ — memory optimization for classes with many instances
import sys

class NormalPoint:
    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z

class SlottedPoint:
    __slots__ = ("x", "y", "z")

    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z


np_ = NormalPoint(1.0, 2.0, 3.0)
sp_ = SlottedPoint(1.0, 2.0, 3.0)

print(f"Normal point size:  {sys.getsizeof(np_)} bytes + dict: {sys.getsizeof(np_.__dict__)} bytes")
print(f"Slotted point size: {sys.getsizeof(sp_)} bytes  (no __dict__)")

# Verify slotted classes have no __dict__
print(f"\nHas __dict__: NormalPoint={hasattr(np_, '__dict__')}, SlottedPoint={hasattr(sp_, '__dict__')}")

# Simulate 1M instances
import tracemalloc
tracemalloc.start()
normals = [NormalPoint(i, i, i) for i in range(100_000)]
snapshot = tracemalloc.take_snapshot()
normal_mem = sum(stat.size for stat in snapshot.statistics("lineno"))

tracemalloc.clear_traces()
slotted = [SlottedPoint(i, i, i) for i in range(100_000)]
snapshot2 = tracemalloc.take_snapshot()
slot_mem = sum(stat.size for stat in snapshot2.statistics("lineno"))
tracemalloc.stop()

print(f"\n100k instances — Normal: {normal_mem/1024:.0f} KB | Slotted: {slot_mem/1024:.0f} KB")
print(f"Memory saved: ~{(1 - slot_mem/normal_mem)*100:.0f}%")


In [ ]:
# __init_subclass__ — hook called when a class is subclassed
class Registry:
    """Auto-registers all subclasses by their 'name' attribute."""
    _registry: dict = {}

    def __init_subclass__(cls, name: str = None, **kwargs):
        super().__init_subclass__(**kwargs)
        if name:
            Registry._registry[name] = cls
            print(f"  Registered plugin: {name!r} -> {cls.__name__}")

    @classmethod
    def get(cls, name: str):
        return cls._registry.get(name)


print("Defining plugins...")

class JSONPlugin(Registry, name="json"):
    def process(self, data): return f"JSON: {data}"

class CSVPlugin(Registry, name="csv"):
    def process(self, data): return f"CSV: {data}"

class XMLPlugin(Registry, name="xml"):
    def process(self, data): return f"XML: {data}"

print(f"\nAll registered: {list(Registry._registry.keys())}")
plugin = Registry.get("csv")()
print(plugin.process("hello"))


## 4. Design Patterns: Singleton, Factory, Observer

Design patterns are reusable solutions to common design problems.


In [ ]:
# ── SINGLETON PATTERN ──
# Ensures only ONE instance of a class ever exists.
# Thread-safe version using a metaclass.

import threading

class SingletonMeta(type):
    _instances = {}
    _lock = threading.Lock()

    def __call__(cls, *args, **kwargs):
        with cls._lock:
            if cls not in cls._instances:
                instance = super().__call__(*args, **kwargs)
                cls._instances[cls] = instance
        return cls._instances[cls]


class DatabaseConnection(metaclass=SingletonMeta):
    def __init__(self, url: str = "sqlite:///app.db"):
        self.url = url
        self.connected = False
        print(f"  Creating connection to {url}")

    def connect(self):
        self.connected = True
        return self

    def __repr__(self):
        return f"DBConnection(url={self.url!r}, id={id(self)})"


print("Creating connections...")
db1 = DatabaseConnection("sqlite:///app.db")
db2 = DatabaseConnection("sqlite:///other.db")  # Ignored — returns same instance

print(f"\ndb1 is db2: {db1 is db2}")   # True — same object
print(f"db1: {db1}")
print(f"db2: {db2}")  # Same URL — second __init__ never called


In [ ]:
# ── FACTORY PATTERN ──
# Creates objects without specifying the exact class.

from abc import ABC, abstractmethod

class Notification(ABC):
    @abstractmethod
    def send(self, message: str, recipient: str) -> str: ...

class EmailNotification(Notification):
    def send(self, message: str, recipient: str) -> str:
        return f"📧 EMAIL to {recipient}: {message}"

class SMSNotification(Notification):
    def send(self, message: str, recipient: str) -> str:
        return f"📱 SMS to {recipient}: {message}"

class SlackNotification(Notification):
    def send(self, message: str, recipient: str) -> str:
        return f"💬 SLACK @{recipient}: {message}"

class PushNotification(Notification):
    def send(self, message: str, recipient: str) -> str:
        return f"🔔 PUSH to {recipient}: {message}"


class NotificationFactory:
    _channels: dict[str, type[Notification]] = {
        "email": EmailNotification,
        "sms": SMSNotification,
        "slack": SlackNotification,
        "push": PushNotification,
    }

    @classmethod
    def create(cls, channel: str) -> Notification:
        if channel not in cls._channels:
            raise ValueError(f"Unknown channel: {channel!r}. Choose from {list(cls._channels)}")
        return cls._channels[channel]()

    @classmethod
    def register(cls, channel: str, klass: type[Notification]):
        """Extend the factory with new channels at runtime."""
        cls._channels[channel] = klass


# Usage
for channel in ["email", "sms", "slack", "push"]:
    notifier = NotificationFactory.create(channel)
    print(notifier.send("Your order shipped!", "alice"))


In [ ]:
# ── OBSERVER PATTERN ──
# Defines a one-to-many dependency: when one object changes state,
# all dependents are notified automatically.

from abc import ABC, abstractmethod
from typing import Any
from dataclasses import dataclass, field

class Observer(ABC):
    @abstractmethod
    def update(self, event: str, data: Any): ...

class EventBus:
    """Subject/Publisher — maintains a list of observers per event."""

    def __init__(self):
        self._listeners: dict[str, list[Observer]] = {}

    def subscribe(self, event: str, observer: Observer):
        self._listeners.setdefault(event, []).append(observer)
        return self  # Fluent API

    def unsubscribe(self, event: str, observer: Observer):
        if event in self._listeners:
            self._listeners[event].remove(observer)

    def publish(self, event: str, data: Any = None):
        for obs in self._listeners.get(event, []):
            obs.update(event, data)


# Concrete observers
class EmailAlert(Observer):
    def update(self, event, data):
        print(f"  📧 EmailAlert [{event}]: sending email — {data}")

class AuditLog(Observer):
    def __init__(self):
        self.log = []
    def update(self, event, data):
        entry = f"[{event}] {data}"
        self.log.append(entry)
        print(f"  📋 AuditLog: recorded — {entry}")

class Dashboard(Observer):
    def update(self, event, data):
        print(f"  📊 Dashboard: refreshing widget for {event}")


# Wire it up
bus = EventBus()
email = EmailAlert()
audit = AuditLog()
dashboard = Dashboard()

bus.subscribe("order.placed", email)    .subscribe("order.placed", audit)    .subscribe("order.placed", dashboard)    .subscribe("payment.failed", email)    .subscribe("payment.failed", audit)

print("=== Order Placed ===")
bus.publish("order.placed", {"order_id": "ORD-001", "total": 49.99})

print("\n=== Payment Failed ===")
bus.publish("payment.failed", {"order_id": "ORD-002", "reason": "Insufficient funds"})

print(f"\nAudit log entries: {audit.log}")


## 5. Dataclasses — Advanced Usage

`@dataclass` eliminates boilerplate while offering powerful customization via `field()`, `__post_init__`, frozen mode, and more.


In [ ]:
from dataclasses import dataclass, field, KW_ONLY
from typing import ClassVar
from datetime import datetime

@dataclass
class Address:
    street: str
    city: str
    country: str = "India"

    def __str__(self):
        return f"{self.street}, {self.city}, {self.country}"


@dataclass(order=True, frozen=False)
class Employee:
    # ClassVar doesn't become an instance field
    _count: ClassVar[int] = 0

    # sort_index is used for comparisons (order=True)
    sort_index: float = field(init=False, repr=False)

    # KW_ONLY — everything after this must be keyword-only
    _: KW_ONLY
    name: str
    department: str
    salary: float = 50_000.0
    skills: list[str] = field(default_factory=list)
    address: Address = field(default_factory=lambda: Address("Unknown", "Unknown"))
    created_at: datetime = field(default_factory=datetime.now, repr=False)

    def __post_init__(self):
        Employee._count += 1
        object.__setattr__(self, "sort_index", self.salary)  # sort by salary
        if self.salary < 0:
            raise ValueError(f"Salary cannot be negative: {self.salary}")

    @classmethod
    def total_employees(cls) -> int:
        return cls._count

    def give_raise(self, percent: float):
        self.salary *= (1 + percent / 100)
        self.sort_index = self.salary  # keep sort index in sync


# Create employees
e1 = Employee(name="Priya", department="Engineering", salary=90_000, skills=["Python", "FastAPI"])
e2 = Employee(name="Rohan", department="Design", salary=70_000, skills=["Figma", "CSS"])
e3 = Employee(name="Arjun", department="Engineering", salary=110_000, skills=["Python", "AWS", "K8s"])

print(e1)
print(e2)
print()

# Comparison works because order=True
employees = sorted([e1, e2, e3])
print("Sorted by salary:")
for e in employees:
    print(f"  {e.name}: ₹{e.salary:,.0f}")

print(f"\nTotal employees: {Employee.total_employees()}")


In [ ]:
# Frozen dataclass — immutable, usable as dict key / set member
from dataclasses import dataclass

@dataclass(frozen=True)
class Point:
    x: float
    y: float

    def distance_to(self, other: "Point") -> float:
        return ((self.x - other.x)**2 + (self.y - other.y)**2) ** 0.5


p1 = Point(0, 0)
p2 = Point(3, 4)
print(f"Distance: {p1.distance_to(p2)}")

# Can be used as dict keys and in sets (it's hashable)
points = {p1, p2, Point(0, 0)}  # duplicate Point(0,0) removed
print(f"Unique points: {points}")

cache = {p1: "origin", p2: "3-4-5"}
print(f"Cache lookup: {cache[Point(0, 0)]}")  # Works!

# Frozen = immutable
try:
    p1.x = 5
except Exception as e:
    print(f"\nCannot mutate frozen dataclass: {e}")


---
## ✅ Quiz — Test Your Understanding

Answer these before looking at the solutions below!


**Q1.** What does MRO stand for and which algorithm does Python use for it?

**Q2.** When would you use `__slots__` over a regular class attribute?

**Q3.** What is the key difference between a `@classmethod` and a `@staticmethod`?

**Q4.** In the Observer pattern, what is the role of the "Subject"?

**Q5.** What does `field(default_factory=list)` solve that `field(default=[])`  doesn't?

---
### 💡 Solutions


In [ ]:
# Q1: MRO — Method Resolution Order
# Python uses the C3 Linearization algorithm
print("Q1:", [c.__name__ for c in list.__mro__])

# Q2: __slots__ saves memory — no per-instance __dict__
# Use when creating 10,000+ instances of a class

# Q3: classmethod vs staticmethod
class Demo:
    value = 42

    @classmethod
    def cls_method(cls):
        return f"classmethod receives cls={cls.__name__}, can access cls.value={cls.value}"

    @staticmethod
    def static_method():
        return "staticmethod receives nothing implicitly"

print("Q3a:", Demo.cls_method())
print("Q3b:", Demo.static_method())

# Q4: In Observer, the Subject (EventBus) maintains subscriber list and notifies them

# Q5: Mutable default argument bug!
from dataclasses import dataclass, field

# WRONG — all instances share the same list!
# @dataclass
# class Bad:
#     tags: list = []   # This is actually an error in dataclasses — raises ValueError

# CORRECT — each instance gets a fresh list
@dataclass
class Good:
    tags: list = field(default_factory=list)

a, b = Good(), Good()
a.tags.append("python")
print(f"\nQ5 — a.tags={a.tags}, b.tags={b.tags}")  # b.tags is empty — not shared!


---
## 💼 Interview Questions & Answers


In [ ]:
# ─────────────────────────────────────────────────────────────
# INTERVIEW Q1: Explain the Liskov Substitution Principle (LSP)
# with a Python example
# ─────────────────────────────────────────────────────────────

# LSP: Subclasses must be substitutable for their base class
# without breaking the program.

class Bird(ABC):
    @abstractmethod
    def move(self) -> str: ...

class FlyingBird(Bird):
    def move(self) -> str:
        return "flying"

class Penguin(Bird):
    """Penguins can't fly — they swim instead."""
    def move(self) -> str:
        return "swimming"   # Still valid — LSP satisfied

# BAD design (violates LSP):
# class Penguin(FlyingBird):
#     def fly(self): raise NotImplementedError("Can't fly!")
# ^ Breaks callers that expect all FlyingBirds to fly

# GOOD design — right abstraction level
birds = [FlyingBird(), Penguin()]
for bird in birds:
    print(f"{type(bird).__name__}: {bird.move()}")  # All work, no surprises


In [ ]:
# ─────────────────────────────────────────────────────────────
# INTERVIEW Q2: Thread-safe Singleton
# ─────────────────────────────────────────────────────────────
import threading

class ThreadSafeSingleton:
    _instance = None
    _lock = threading.Lock()

    def __new__(cls):
        if cls._instance is None:
            with cls._lock:
                # Double-checked locking
                if cls._instance is None:
                    cls._instance = super().__new__(cls)
        return cls._instance

# Test with concurrent threads
results = []
def create():
    results.append(id(ThreadSafeSingleton()))

threads = [threading.Thread(target=create) for _ in range(50)]
for t in threads: t.start()
for t in threads: t.join()

unique_ids = set(results)
print(f"50 threads created {len(unique_ids)} unique instance(s) — thread-safe: {len(unique_ids) == 1}")


In [ ]:
# ─────────────────────────────────────────────────────────────
# INTERVIEW Q3: Composition over Inheritance
# ─────────────────────────────────────────────────────────────

# Inheritance: "IS-A" relationship
# Composition: "HAS-A" relationship — more flexible, easier to test

# Inheritance approach — rigid, hard to mix features
class LoggingMixin:
    def log(self, msg): print(f"[LOG] {msg}")

class CachingMixin:
    _cache = {}
    def cached_call(self, key, fn):
        if key not in self._cache:
            self._cache[key] = fn()
        return self._cache[key]

# Composition approach — flexible, each piece independently testable
class Logger:
    def log(self, msg: str, level: str = "INFO"):
        print(f"[{level}] {msg}")

class Cache:
    def __init__(self): self._store = {}
    def get(self, key): return self._store.get(key)
    def set(self, key, val): self._store[key] = val

class UserService:
    """Has-A logger and cache, doesn't inherit from them."""
    def __init__(self):
        self.logger = Logger()
        self.cache = Cache()

    def get_user(self, user_id: int):
        cached = self.cache.get(user_id)
        if cached:
            self.logger.log(f"Cache HIT for user {user_id}")
            return cached
        self.logger.log(f"Cache MISS — fetching user {user_id}", "DEBUG")
        user = {"id": user_id, "name": f"User_{user_id}"}
        self.cache.set(user_id, user)
        return user

svc = UserService()
print(svc.get_user(42))
print(svc.get_user(42))  # Cache hit


---
## 🔨 Real-World Use Case: Pluggable Notification System

Build a production-grade notification system using **Strategy + Factory + ABC**.  
New channels can be added without modifying existing code (Open/Closed Principle).


In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Protocol
import time

# ── Domain objects ──
@dataclass
class Notification:
    recipient: str
    subject: str
    body: str
    priority: str = "normal"   # normal | high | critical


# ── Strategy: each channel is a separate strategy ──
class NotificationChannel(ABC):
    @abstractmethod
    def send(self, notification: Notification) -> bool:
        """Returns True if sent successfully."""
        ...

    @property
    @abstractmethod
    def name(self) -> str: ...


class EmailChannel(NotificationChannel):
    @property
    def name(self): return "email"

    def send(self, n: Notification) -> bool:
        # Simulated send
        time.sleep(0.01)
        print(f"  📧 EMAIL → {n.recipient} | Subject: {n.subject}")
        return True


class SMSChannel(NotificationChannel):
    @property
    def name(self): return "sms"

    def send(self, n: Notification) -> bool:
        if len(n.body) > 160:
            print(f"  ⚠️  SMS body truncated to 160 chars for {n.recipient}")
        print(f"  📱 SMS → {n.recipient} | {n.body[:160]}")
        return True


class SlackChannel(NotificationChannel):
    def __init__(self, workspace: str = "my-workspace"):
        self.workspace = workspace

    @property
    def name(self): return "slack"

    def send(self, n: Notification) -> bool:
        print(f"  💬 SLACK [{self.workspace}] → @{n.recipient} | {n.subject}: {n.body}")
        return True


# ── Factory + Registry ──
class ChannelFactory:
    _registry: dict[str, NotificationChannel] = {}

    @classmethod
    def register(cls, channel: NotificationChannel):
        cls._registry[channel.name] = channel
        return cls  # fluent

    @classmethod
    def get(cls, name: str) -> NotificationChannel:
        if name not in cls._registry:
            raise ValueError(f"Channel {name!r} not registered. Available: {list(cls._registry)}")
        return cls._registry[name]

    @classmethod
    def all_channels(cls) -> list[str]:
        return list(cls._registry.keys())


# ── Notification Service — orchestrates everything ──
class NotificationService:
    def __init__(self):
        self._rules: dict[str, list[str]] = {
            "normal":   ["email"],
            "high":     ["email", "slack"],
            "critical": ["email", "sms", "slack"],
        }

    def send(self, notification: Notification) -> dict[str, bool]:
        channels = self._rules.get(notification.priority, ["email"])
        results = {}
        for ch_name in channels:
            try:
                channel = ChannelFactory.get(ch_name)
                results[ch_name] = channel.send(notification)
            except Exception as e:
                print(f"  ❌ Failed to send via {ch_name}: {e}")
                results[ch_name] = False
        return results


# ── Wire it up ──
ChannelFactory     .register(EmailChannel())     .register(SMSChannel())     .register(SlackChannel("dev-team"))

service = NotificationService()

print("=== Normal priority notification ===")
r1 = service.send(Notification("user@example.com", "Welcome!", "Thanks for signing up."))
print(f"Results: {r1}\n")

print("=== High priority notification ===")
r2 = service.send(Notification("admin", "Disk space low", "Server disk at 90%", priority="high"))
print(f"Results: {r2}\n")

print("=== CRITICAL notification ===")
r3 = service.send(Notification("+91-9999999999", "🚨 DB DOWN", "Database connection lost!", priority="critical"))
print(f"Results: {r3}")


---
## 🚀 Mini Project: Shape Area Calculator

A complete OOP hierarchy using Abstract Base Classes, dataclasses, and a Factory.  
Demonstrates everything covered today in one cohesive example.


In [ ]:
import math
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import ClassVar

# ── Abstract Base ──
class Shape(ABC):
    _count: ClassVar[int] = 0

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        # Auto-register every concrete subclass
        if not getattr(cls, "__abstractmethods__", None):
            ShapeFactory._shapes[cls.__name__.lower()] = cls

    @abstractmethod
    def area(self) -> float: ...

    @abstractmethod
    def perimeter(self) -> float: ...

    @abstractmethod
    def scale(self, factor: float) -> "Shape":
        """Return a new scaled copy."""
        ...

    def describe(self) -> str:
        return (f"{self.__class__.__name__:12s} | "
                f"area={self.area():>10.4f} | "
                f"perimeter={self.perimeter():>10.4f}")

    def __lt__(self, other: "Shape"):
        return self.area() < other.area()


# ── Concrete shapes ──
@dataclass
class Circle(Shape):
    radius: float

    def area(self): return math.pi * self.radius ** 2
    def perimeter(self): return 2 * math.pi * self.radius
    def scale(self, f): return Circle(self.radius * f)


@dataclass
class Rectangle(Shape):
    width: float
    height: float

    def area(self): return self.width * self.height
    def perimeter(self): return 2 * (self.width + self.height)
    def scale(self, f): return Rectangle(self.width * f, self.height * f)

    @property
    def is_square(self): return math.isclose(self.width, self.height)


@dataclass
class Triangle(Shape):
    a: float   # side a
    b: float   # side b
    c: float   # side c

    def __post_init__(self):
        if not (self.a + self.b > self.c and
                self.b + self.c > self.a and
                self.a + self.c > self.b):
            raise ValueError(f"Invalid triangle sides: {self.a}, {self.b}, {self.c}")

    def area(self):
        s = self.perimeter() / 2
        return math.sqrt(s * (s-self.a) * (s-self.b) * (s-self.c))

    def perimeter(self): return self.a + self.b + self.c
    def scale(self, f): return Triangle(self.a*f, self.b*f, self.c*f)


@dataclass
class RegularPolygon(Shape):
    sides: int
    side_length: float

    def __post_init__(self):
        if self.sides < 3:
            raise ValueError("Polygon must have at least 3 sides")

    def area(self):
        return (self.sides * self.side_length**2) / (4 * math.tan(math.pi / self.sides))

    def perimeter(self): return self.sides * self.side_length
    def scale(self, f): return RegularPolygon(self.sides, self.side_length * f)


# ── Factory ──
class ShapeFactory:
    _shapes: dict[str, type[Shape]] = {}

    @classmethod
    def create(cls, shape_type: str, **kwargs) -> Shape:
        key = shape_type.lower()
        if key not in cls._shapes:
            raise ValueError(f"Unknown shape: {shape_type!r}. Available: {list(cls._shapes)}")
        return cls._shapes[key](**kwargs)

    @classmethod
    def available(cls): return list(cls._shapes.keys())


# ── Canvas — manages a collection of shapes ──
class Canvas:
    def __init__(self):
        self.shapes: list[Shape] = []

    def add(self, shape: Shape):
        self.shapes.append(shape)
        return self

    def total_area(self) -> float:
        return sum(s.area() for s in self.shapes)

    def largest(self) -> Shape:
        return max(self.shapes)

    def summary(self):
        print(f"{'Shape':<12} | {'Area':>12} | {'Perimeter':>12}")
        print("─" * 42)
        for s in sorted(self.shapes, reverse=True):
            print(s.describe())
        print("─" * 42)
        print(f"{'TOTAL AREA':<12} | {self.total_area():>12.4f}")
        print(f"Largest: {self.largest().__class__.__name__}")


# ── Demo ──
canvas = Canvas()
canvas     .add(ShapeFactory.create("circle", radius=5))     .add(ShapeFactory.create("rectangle", width=4, height=6))     .add(ShapeFactory.create("triangle", a=3, b=4, c=5))     .add(ShapeFactory.create("regularpolygon", sides=6, side_length=4))     .add(ShapeFactory.create("circle", radius=2.5))

print("=== Shape Canvas Summary ===")
canvas.summary()

print(f"\nAvailable shapes in factory: {ShapeFactory.available()}")

# Scaling
original = ShapeFactory.create("circle", radius=3)
scaled = original.scale(2)
print(f"\nScaled circle: radius {original.radius} → {scaled.radius}")
print(f"Area ratio: {scaled.area() / original.area():.1f}x (should be 4.0x)")


---
## 📚 Day 1 Summary

| Concept | Key Takeaway |
|---|---|
| **Abstract Base Classes** | Enforce contracts — prevent incomplete subclasses |
| **MRO / C3 Linearization** | Python searches left-to-right, depth-first, no repeat |
| **Descriptors** | Power behind `property`, `classmethod` — define `__get__/__set__` |
| **`__slots__`** | Removes `__dict__` per instance — saves ~40% memory |
| **`__init_subclass__`** | Hook fired when a subclass is defined — great for registries |
| **Singleton** | One instance via metaclass + threading.Lock |
| **Factory** | Create objects by name — decouple creation from usage |
| **Observer** | Pub/sub event bus — decouple producers from consumers |
| **Dataclasses** | `field()`, `__post_init__`, `frozen=True`, `order=True` |

---
## 🔜 Tomorrow: Day 2 — Generators, Iterators & Lazy Evaluation

You'll learn how Python handles infinite sequences, memory-efficient pipelines,  
and the full `itertools` toolkit. See you then!
